<a href="https://colab.research.google.com/github/karthik-srivathsa-05/flyrank-ai/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds and audits the feature vector for the Search Intelligence / Content Review Prioritization lane.

The decision moment is the end of February 2026. Features are constructed only from information available during the February 2026 feature window.

The purpose of this notebook is to:
1. Build a small leakage-safe feature vector.
2. Document what every feature means and when it becomes available.
3. Actively test for future-window, label-derived, and product-flag leakage.
4. Record fields deliberately excluded from modeling.

The feature vector is intended for decision support: identifying pages that may deserve content review. It does not establish causal effects on search rankings or traffic.

## 1. Build the feature vector

I use five features for the February 2026 decision window.

The features describe recent search visibility, search traffic, observed position, consistency of visibility, and content freshness.

All features are calculated from February 2026 information only.

The five features are:

1. `impressions_30d` — total observed Google Search impressions during February.
2. `clicks_30d` — total observed Google Search clicks during February.
3. `avg_position` — average observed search position during February.
4. `days_with_impressions` — number of February observations with positive impressions.
5. `days_since_last_update` — number of days between the content's latest update date and the February 28 decision cutoff.

Identifiers such as client and content hashes are retained only for joins and traceability and are not used as model features.

In [19]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from IPython.display import display


def get_hf_token():
    token = os.environ.get("HF_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")

        if token:
            return token

    except Exception:
        pass

    return getpass.getpass("Enter Hugging Face READ token: ")


HF_TOKEN = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet'"
    f")"
)

FACT_MAR = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

DIM_CONTENT = (
    f"read_parquet("
    f"'{REL}/dim_content.parquet'"
    f")"
)

DIM_CLIENTS = (
    f"read_parquet("
    f"'{REL}/dim_clients.parquet'"
    f")"
)

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Decision cutoff: 2026-02-28")
print("Label window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Decision cutoff: 2026-02-28
Label window: March 2026


In [20]:
feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(COALESCE(gsc_impressions, 0)) AS impressions_30d,

        SUM(COALESCE(gsc_clicks, 0)) AS clicks_30d,

        AVG(gsc_avg_position) AS avg_position,

        COUNT(
            CASE
                WHEN gsc_data_available IS TRUE
                 AND COALESCE(gsc_impressions, 0) > 0
                THEN 1
            END
        ) AS days_with_impressions

    FROM {FACT_FEB}

    WHERE report_date BETWEEN DATE '2026-02-01'
                          AND DATE '2026-02-28'

    GROUP BY
        client_hash_id,
        content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date
    FROM {DIM_CONTENT}
)

SELECT
    feb.client_hash_id,
    feb.content_hash_id,

    feb.impressions_30d,
    feb.clicks_30d,
    feb.avg_position,
    feb.days_with_impressions,

    CASE
        WHEN content.content_updated_date IS NULL
        THEN NULL
        ELSE DATE_DIFF(
            'day',
            content.content_updated_date,
            DATE '2026-02-28'
        )
    END AS days_since_last_update

FROM feb

LEFT JOIN content
    ON feb.client_hash_id = content.client_hash_id
    AND feb.content_hash_id = content.content_hash_id
""").df()


print("Feature frame created successfully.")
print("Number of feature rows:", len(feature_frame))
print("Number of columns:", len(feature_frame.columns))

display(feature_frame.head())

Feature frame created successfully.
Number of feature rows: 321546
Number of columns: 7


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position,days_with_impressions,days_since_last_update
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,0,-81
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,0,-81
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,0,-81
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,0,-81
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,0,-81


In [21]:
print("Number of feature rows:", len(feature_frame))
print("Number of columns:", len(feature_frame.columns))

print("\nMissing values:")
display(feature_frame.isna().sum().to_frame("missing_count"))

print("\nDuplicate client-content observations:")
duplicate_features = (
    feature_frame
    .groupby(["client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
)

duplicate_features = duplicate_features[
    duplicate_features["row_count"] > 1
]

print(len(duplicate_features))
display(duplicate_features.head(10))

Number of feature rows: 321546
Number of columns: 7

Missing values:


,missing_count
client_hash_id,0
content_hash_id,0
impressions_30d,0
clicks_30d,0
avg_position,167987
days_with_impressions,0
days_since_last_update,0



Duplicate client-content observations:
0


,client_hash_id,content_hash_id,row_count


## 2. Feature notes

| Feature | Meaning | Missing-value handling | Available when? |
|---|---|---|---|
| `impressions_30d` | Total observed search impressions during February | Missing daily impressions are treated as zero | Available by 2026-02-28 |
| `clicks_30d` | Total observed search clicks during February | Missing daily clicks are treated as zero | Available by 2026-02-28 |
| `avg_position` | Mean observed search position across February observations where position exists | Kept missing when no position observation exists | Available by 2026-02-28 |
| `days_with_impressions` | Number of February observations with positive impressions | No missing value after aggregation | Available by 2026-02-28 |
| `days_since_last_update` | Days from the latest recorded content update to the February 28 cutoff | Missing when no update date is available | Available by 2026-02-28 when update metadata exists |

The feature vector contains no March outcome information.

The features are therefore intended to represent information available at the February decision moment rather than information learned after the decision.

In [22]:
feature_columns = [
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "days_with_impressions",
    "days_since_last_update"
]

feature_summary = pd.DataFrame({
    "feature": feature_columns,
    "dtype": [
        str(feature_frame[c].dtype)
        for c in feature_columns
    ],
    "missing_count": [
        int(feature_frame[c].isna().sum())
        for c in feature_columns
    ],
    "available_rows": [
        int(feature_frame[c].notna().sum())
        for c in feature_columns
    ]
})

display(feature_summary)

,feature,dtype,missing_count,available_rows
0,impressions_30d,float64,0,321546
1,clicks_30d,float64,0,321546
2,avg_position,float64,167987,153559
3,days_with_impressions,int64,0,321546
4,days_since_last_update,int64,0,321546


## 3. The leakage hunt

I treat leakage as any information that would not have been available at the February 28 decision moment but is allowed to influence the feature vector.

I check three categories:

1. Future-window leakage — March data must not enter the February features.
2. Label-derived leakage — the future outcome/label must not be present in the feature columns.
3. Product-flag leakage — downstream action flags or other outcome-derived recommendations must not be used as features.

The goal is not only to avoid leakage accidentally, but to demonstrate that the feature frame contains only decision-time information.

In [23]:
# Test that the feature frame itself contains no March-derived columns.

future_terms = [
    "march",
    "future",
    "label",
    "outcome",
    "target",
    "decline"
]

future_like_columns = [
    c for c in feature_frame.columns
    if any(term in c.lower() for term in future_terms)
]

print("Potential future/label-derived feature columns:")
print(future_like_columns)

assert len(future_like_columns) == 0, (
    "Potential future or label-derived column found in feature frame."
)

print("PASS: no future/label-derived columns are present.")

Potential future/label-derived feature columns:
[]
PASS: no future/label-derived columns are present.


In [24]:
feb_date_check = con.sql(f"""
SELECT
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(*) AS row_count
FROM {FACT_FEB}
WHERE report_date BETWEEN DATE '2026-02-01'
                      AND DATE '2026-02-28'
""").df()

display(feb_date_check)

assert str(feb_date_check.loc[0, "min_report_date"])[:10] == "2026-02-01"
assert str(feb_date_check.loc[0, "max_report_date"])[:10] == "2026-02-28"

print("PASS: performance features use the February decision window only.")

,min_report_date,max_report_date,row_count
0,2026-02-01,2026-02-28,7355108


PASS: performance features use the February decision window only.


In [25]:
# March exists in the warehouse, but it is intentionally not joined
# into the feature construction.

march_rows = con.sql(f"""
SELECT COUNT(*) AS march_rows
FROM {FACT_FEB}
WHERE report_date >= DATE '2026-03-01'
""").df()

display(march_rows)

print(
    "March rows are not used in feature construction. "
    "They are reserved for future outcome/validation logic."
)

,march_rows
0,0


March rows are not used in feature construction. They are reserved for future outcome/validation logic.


In [26]:
# ---------------------------------------------------------
# 3. Leakage hunt — inspect the actual warehouse schema
# ---------------------------------------------------------

schema_check = con.sql(f"""
DESCRIBE SELECT *
FROM {FACT_FEB}
""").df()

display(schema_check)


# Get all available February fact-table columns
all_fact_columns = schema_check["column_name"].tolist()

print("Number of fact-table columns:", len(all_fact_columns))
print("\nFact-table columns:")
print(all_fact_columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Number of fact-table columns: 31

Fact-table columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [27]:
# ---------------------------------------------------------
# Identify potentially dangerous flag / product columns
# ---------------------------------------------------------

flag_like_columns = [
    c for c in all_fact_columns
    if any(
        term in c.lower()
        for term in [
            "flag",
            "label",
            "outcome",
            "action",
            "recommendation",
            "product"
        ]
    )
]

print("Potential flag/label/product columns:")
print(flag_like_columns)

Potential flag/label/product columns:
[]


In [28]:
# ---------------------------------------------------------
# Check that our five features are not future/label fields
# ---------------------------------------------------------

feature_columns = [
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "days_with_impressions",
    "days_since_last_update"
]

future_terms = [
    "march",
    "future",
    "label",
    "outcome",
    "flag",
    "action",
    "recommendation"
]

feature_leakage_candidates = [
    c for c in feature_columns
    if any(term in c.lower() for term in future_terms)
]

print("Feature columns:", feature_columns)
print("Potential leakage candidates:", feature_leakage_candidates)

assert len(feature_leakage_candidates) == 0, (
    "Potential leakage detected in feature names."
)

print("Leakage name check: PASS")

Feature columns: ['impressions_30d', 'clicks_30d', 'avg_position', 'days_with_impressions', 'days_since_last_update']
Potential leakage candidates: []
Leakage name check: PASS


In [29]:
# ---------------------------------------------------------
# Final feature-frame check
# ---------------------------------------------------------

model_features = [
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "days_with_impressions",
    "days_since_last_update"
]

missing_features = [
    c for c in model_features
    if c not in feature_frame.columns
]

print("Expected model features:")
print(model_features)

print("\nMissing expected features:")
print(missing_features)

assert len(missing_features) == 0, (
    f"Missing features: {missing_features}"
)

print("\nFeature vector check: PASS")

Expected model features:
['impressions_30d', 'clicks_30d', 'avg_position', 'days_with_impressions', 'days_since_last_update']

Missing expected features:
[]

Feature vector check: PASS


In [30]:
excluded_flag_columns = [
    c for c in feature_frame.columns
    if c in flag_like_columns
]

print("Product/action flag columns present in feature frame:")
print(excluded_flag_columns)

assert len(excluded_flag_columns) == 0

print("PASS: no product/action flag columns are used as features.")

Product/action flag columns present in feature frame:
[]
PASS: no product/action flag columns are used as features.


## 4. What I excluded and why

I deliberately exclude the following fields from the model feature vector:

| Excluded field/category | Why excluded |
|---|---|
| `client_hash_id` | Entity identifier; useful for grouping and joins but not a meaningful predictive feature. |
| `content_hash_id` | Entity identifier; retained for traceability and ranking output but not used as a feature. |
| March performance metrics | Future information relative to the February 28 decision moment. |
| Future labels/outcomes | These are only known after the decision and would directly leak the target. |
| Product/action flags | These represent downstream decisions or derived signals and could leak the intended action into the model. |
| June 2026 data | The final month is treated as a sealed outcome/test period rather than development data. |
| Client names, URLs, or other identifying information | Not required for the public-safe modeling task and deliberately excluded for privacy. |

The feature vector is intentionally small. The objective is to establish a reproducible, interpretable baseline feature set rather than maximize the number of available columns.

The resulting features are decision-time signals, not proof of causality.

In [31]:
MODEL_FEATURES = [
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "days_with_impressions",
    "days_since_last_update"
]

print("Final model features:")
for i, feature in enumerate(MODEL_FEATURES, start=1):
    print(f"{i}. {feature}")

excluded_from_model = [
    "client_hash_id",
    "content_hash_id",
    "March performance metrics",
    "future labels/outcomes",
    "product/action flags",
    "June 2026 data",
    "client names / URLs / identifying information"
]

print("\nExcluded:")
for item in excluded_from_model:
    print("-", item)

Final model features:
1. impressions_30d
2. clicks_30d
3. avg_position
4. days_with_impressions
5. days_since_last_update

Excluded:
- client_hash_id
- content_hash_id
- March performance metrics
- future labels/outcomes
- product/action flags
- June 2026 data
- client names / URLs / identifying information


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.